# Treino CatBoost para Cascata Anti-Phishing

Modelo de segundo estágio da arquitetura em cascata:
- **Estágio 1:** DomURLs-BERT classifica a URL bruta (~20ms)
- **Estágio 2 (este modelo):** CatBoost com features estruturadas (WHOIS, DNS, TLS) para URLs incertas

## Requisitos
- Google Colab (T4 não necessário, CPU é suficiente)
- Upload: `dataset_phishing_brasileiro.csv` (da pasta `benchmark-modelos/`)
- Tempo estimado: ~2-4h (extração de features para 20k URLs)

In [ ]:
# Cel 1 - Setup
!pip install -q catboost python-whois dnspython httpx scikit-learn

import pandas as pd
import numpy as np
import json
import time
import asyncio
import ssl
import socket
import re
import logging
from datetime import datetime, timezone
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
print("Setup conclu\u00eddo!")

In [ ]:
# Cel 2 - Carregar dataset e amostrar 20k URLs (10k legit + 10k phish)
from google.colab import files

# Upload do dataset
print("Fa\u00e7a upload do dataset_phishing_brasileiro.csv")
uploaded = files.upload()
DATASET_FILE = list(uploaded.keys())[0]

df_full = pd.read_csv(DATASET_FILE)
print(f"Dataset completo: {len(df_full):,} URLs")
print(f"Distribui\u00e7\u00e3o:\n{df_full['label'].value_counts()}")

# Amostrar subset balanceado
SAMPLE_PER_CLASS = 10_000
df_legit = df_full[df_full['label'] == 0].sample(n=SAMPLE_PER_CLASS, random_state=42)
df_phish = df_full[df_full['label'] == 1].sample(n=SAMPLE_PER_CLASS, random_state=42)
df = pd.concat([df_legit, df_phish]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nSubset amostrado: {len(df):,} URLs ({SAMPLE_PER_CLASS:,} por classe)")

In [ ]:
# Cel 3 - Extrair client features (instant, mesma logica da extensao)

def extract_client_features(url: str) -> dict:
    """Extrai features client-side de uma URL (mesma logica da extensao)."""
    try:
        full_url = url if url.startswith("http") else f"http://{url}"
        parsed = urlparse(full_url)
        hostname = parsed.hostname or ""
        vowels = sum(1 for c in hostname if c in "aeiou")
        params = url.count("&") + (1 if "?" in url else 0)
        return {
            "length": len(url),
            "dom_length": len(hostname),
            "dot": url.count("."),
            "hyphen": url.count("-"),
            "slash": url.count("/"),
            "at": url.count("@"),
            "params": params,
            "shortened": 1 if len(hostname) <= 6 and "." in hostname else 0,
            "tls": 1 if parsed.scheme == "https" else 0,
            "vowels_domain": vowels,
            "email": 1 if "@" in url and "." in url.split("@")[-1] else 0,
        }
    except Exception:
        return {
            "length": len(url), "dom_length": 0, "dot": 0, "hyphen": 0,
            "slash": 0, "at": 0, "params": 0, "shortened": 0,
            "tls": 0, "vowels_domain": 0, "email": 0,
        }

# Extrair para todo o subset
client_features = df['url'].apply(extract_client_features).apply(pd.Series)
print(f"Client features extra\u00eddas: {client_features.shape}")
client_features.head()

In [ ]:
# Cel 4 - Funcoes de extracao server-side (WHOIS, DNS, TLS)

import whois
import dns.resolver
import httpx

def extract_domain(url: str) -> str:
    """Extrai dominio de uma URL."""
    try:
        parsed = urlparse(url if "://" in url else f"http://{url}")
        hostname = parsed.hostname or ""
        if hostname.startswith("www."):
            hostname = hostname[4:]
        return hostname.lower()
    except Exception:
        return ""


def lookup_whois(domain: str) -> dict:
    """WHOIS lookup com timeout."""
    try:
        result = whois.whois(domain)
        if not result or not result.domain_name:
            return {"status": "not_found"}

        data = {"status": "found"}
        now = datetime.now(timezone.utc)

        # Idade do dominio
        creation = result.creation_date
        if isinstance(creation, list):
            creation = creation[0]
        if creation:
            if creation.tzinfo is None:
                creation = creation.replace(tzinfo=timezone.utc)
            data["dom_age"] = (now - creation).days

        # Expiracao
        expiration = result.expiration_date
        if isinstance(expiration, list):
            expiration = expiration[0]
        if expiration:
            if expiration.tzinfo is None:
                expiration = expiration.replace(tzinfo=timezone.utc)
            data["dom_expire"] = (expiration - now).days

        # Registrar
        if result.registrar:
            data["registrar"] = str(result.registrar).strip()[:60]

        # Pais
        country = getattr(result, 'country', None)
        if country:
            if isinstance(country, list):
                country = country[0]
            data["country_code"] = str(country).strip().upper()[:2]

        # Privacidade WHOIS
        privacy_keywords = ["privacy", "redacted", "proxy", "whoisguard",
                           "withheld", "private", "protected", "contact privacy"]
        registrant = str(getattr(result, 'name', '') or '')
        org = str(getattr(result, 'org', '') or '')
        whois_str = (registrant + " " + org).lower()
        data["whois_privacy"] = 1 if any(kw in whois_str for kw in privacy_keywords) else 0

        return data
    except Exception:
        return {"status": "error"}


def lookup_dns(domain: str) -> dict:
    """DNS lookups: MX, NS, SPF, A records."""
    result = {"mx_servers": -1, "nameservers": -1, "dom_spf": -1,
              "dom_in_ip": -1, "srv_client": -1}
    resolver = dns.resolver.Resolver()
    resolver.timeout = 3
    resolver.lifetime = 3

    # MX
    try:
        answers = resolver.resolve(domain, "MX")
        result["mx_servers"] = len(list(answers))
    except Exception:
        pass

    # NS
    try:
        answers = resolver.resolve(domain, "NS")
        result["nameservers"] = len(list(answers))
    except Exception:
        pass

    # SPF
    try:
        answers = resolver.resolve(domain, "TXT")
        result["dom_spf"] = 0
        for rdata in answers:
            if "v=spf1" in rdata.to_text().lower():
                result["dom_spf"] = 1
                break
    except Exception:
        pass

    # A record
    try:
        answers = resolver.resolve(domain, "A")
        result["dom_in_ip"] = 1 if re.match(r"^\d+\.\d+\.\d+\.\d+$", domain) else 0
        result["srv_client"] = 1 if list(answers) else 0
    except Exception:
        pass

    return result


def lookup_tls(url: str) -> dict:
    """Extrai informacoes do certificado TLS."""
    result = {"tls_issuer": "unknown", "tls_validity_days": -1, "tls_san_count": -1}
    try:
        full_url = url if url.startswith("http") else f"http://{url}"
        parsed = urlparse(full_url)
        hostname = parsed.hostname or ""
        port = parsed.port or 443

        if parsed.scheme != "https":
            result["tls_issuer"] = "none"
            result["tls_validity_days"] = 0
            result["tls_san_count"] = 0
            return result

        ctx = ssl.create_default_context()
        with socket.create_connection((hostname, port), timeout=5) as sock:
            with ctx.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()

        # Issuer
        issuer_dict = dict(x[0] for x in cert.get("issuer", []))
        issuer_org = issuer_dict.get("organizationName", "unknown")
        result["tls_issuer"] = str(issuer_org)[:40]

        # Validade
        not_after = cert.get("notAfter")
        if not_after:
            expire = datetime.strptime(not_after, "%b %d %H:%M:%S %Y %Z")
            result["tls_validity_days"] = (expire - datetime.utcnow()).days

        # SANs
        san = cert.get("subjectAltName", [])
        result["tls_san_count"] = len(san)

        return result
    except Exception:
        return result


def count_redirects(url: str) -> int:
    """Conta redirects HTTP."""
    try:
        full_url = url if url.startswith("http") else f"http://{url}"
        with httpx.Client(follow_redirects=False, timeout=5, verify=False) as client:
            count = 0
            current = full_url
            for _ in range(10):
                try:
                    resp = client.head(current)
                    if resp.is_redirect and resp.headers.get("location"):
                        count += 1
                        loc = resp.headers["location"]
                        if not loc.startswith("http"):
                            from urllib.parse import urljoin
                            loc = urljoin(current, loc)
                        current = loc
                    else:
                        break
                except Exception:
                    break
            return count
    except Exception:
        return -1


print("Fun\u00e7\u00f5es de extra\u00e7\u00e3o definidas!")

In [ ]:
# Cel 5 - Extrair server features para o subset (paralelizado)
# ATENCAO: esta celula demora ~2-4h para 20k URLs

from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings("ignore")

def extract_all_server_features(url: str) -> dict:
    """Extrai todas as features server-side para uma URL."""
    domain = extract_domain(url)
    if not domain:
        return {"redirects": -1, "dom_age": -1, "dom_expire": -1,
                "registrar": "unknown", "country_code": "unknown",
                "whois_privacy": -1, "mx_servers": -1, "nameservers": -1,
                "dom_spf": -1, "dom_in_ip": -1, "srv_client": -1,
                "tls_issuer": "unknown", "tls_validity_days": -1, "tls_san_count": -1}

    # WHOIS
    w = lookup_whois(domain)
    # DNS
    d = lookup_dns(domain)
    # TLS
    t = lookup_tls(url)
    # Redirects
    r = count_redirects(url)

    return {
        "redirects": r,
        "dom_age": w.get("dom_age", -1),
        "dom_expire": w.get("dom_expire", -1),
        "registrar": w.get("registrar", "unknown"),
        "country_code": w.get("country_code", "unknown"),
        "whois_privacy": w.get("whois_privacy", -1),
        "mx_servers": d.get("mx_servers", -1),
        "nameservers": d.get("nameservers", -1),
        "dom_spf": d.get("dom_spf", -1),
        "dom_in_ip": d.get("dom_in_ip", -1),
        "srv_client": d.get("srv_client", -1),
        "tls_issuer": t.get("tls_issuer", "unknown"),
        "tls_validity_days": t.get("tls_validity_days", -1),
        "tls_san_count": t.get("tls_san_count", -1),
    }

# Cache por dominio (muitas URLs compartilham o mesmo dominio)
domain_cache = {}
results = []
urls = df['url'].tolist()
total = len(urls)

print(f"Extraindo server features para {total:,} URLs...")
print("Isso pode demorar 2-4 horas. Progresso sera mostrado a cada 500 URLs.")
start_time = time.time()

def process_url(idx_url):
    idx, url = idx_url
    domain = extract_domain(url)
    # Cache por dominio
    if domain in domain_cache:
        return idx, domain_cache[domain].copy()
    feats = extract_all_server_features(url)
    domain_cache[domain] = feats
    return idx, feats

server_results = [None] * total
completed = 0

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(process_url, (i, url)): i for i, url in enumerate(urls)}
    for future in as_completed(futures):
        try:
            idx, feats = future.result()
            server_results[idx] = feats
        except Exception as e:
            idx = futures[future]
            server_results[idx] = {"redirects": -1, "dom_age": -1, "dom_expire": -1,
                "registrar": "unknown", "country_code": "unknown",
                "whois_privacy": -1, "mx_servers": -1, "nameservers": -1,
                "dom_spf": -1, "dom_in_ip": -1, "srv_client": -1,
                "tls_issuer": "unknown", "tls_validity_days": -1, "tls_san_count": -1}
        completed += 1
        if completed % 500 == 0:
            elapsed = time.time() - start_time
            rate = completed / elapsed
            eta = (total - completed) / rate if rate > 0 else 0
            print(f"  [{completed:,}/{total:,}] {rate:.1f} URLs/s | ETA: {eta/60:.0f}min")

server_features = pd.DataFrame(server_results)
elapsed = time.time() - start_time
print(f"\nExtra\u00e7\u00e3o conclu\u00edda em {elapsed/60:.1f} minutos!")
print(f"Dominios unicos processados: {len(domain_cache):,}")
server_features.head()

In [ ]:
# Cel 6 - Salvar dataset enriquecido (para nao precisar re-extrair)

df_enriched = pd.concat([
    df[['url', 'label']].reset_index(drop=True),
    client_features.reset_index(drop=True),
    server_features.reset_index(drop=True)
], axis=1)

ENRICHED_FILE = "dataset_cascata_20k.csv"
df_enriched.to_csv(ENRICHED_FILE, index=False)
print(f"Dataset enriquecido salvo: {ENRICHED_FILE}")
print(f"Shape: {df_enriched.shape}")
print(f"Colunas: {list(df_enriched.columns)}")

# Estatisticas de missing (-1 ou 'unknown')
print("\n--- Taxa de valores desconhecidos ---")
for col in server_features.columns:
    if server_features[col].dtype == 'object':
        missing = (server_features[col] == 'unknown').sum()
    else:
        missing = (server_features[col] == -1).sum()
    pct = missing / len(server_features) * 100
    print(f"  {col:20s}: {missing:5d} ({pct:.1f}%)")

# Baixar backup
files.download(ENRICHED_FILE)

In [ ]:
# Cel 7 - Preparar features para treino

from sklearn.model_selection import train_test_split

# Se retomando de um dataset salvo, descomente:
# df_enriched = pd.read_csv("dataset_cascata_20k.csv")

# Features e target
FEATURE_COLS = [
    # Client features (11)
    'length', 'dom_length', 'dot', 'hyphen', 'slash', 'at',
    'params', 'shortened', 'tls', 'vowels_domain', 'email',
    # Server features numericas (7)
    'redirects', 'dom_age', 'dom_expire', 'mx_servers',
    'nameservers', 'dom_spf', 'dom_in_ip',
    'tls_validity_days', 'tls_san_count',
    # Server features categoricas (3)
    'registrar', 'country_code', 'tls_issuer',
    # WHOIS privacy (binaria)
    'whois_privacy',
]

# Features categoricas (CatBoost trata nativamente)
CAT_FEATURES = ['registrar', 'country_code', 'tls_issuer']

X = df_enriched[FEATURE_COLS].copy()
y = df_enriched['label'].copy()

# Preencher NaN em categoricas com 'unknown'
for col in CAT_FEATURES:
    X[col] = X[col].fillna('unknown').astype(str)

# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Features: {len(FEATURE_COLS)} ({len(CAT_FEATURES)} categ\u00f3ricas)")
print(f"\nDistribui\u00e7\u00e3o treino:\n{y_train.value_counts()}")
print(f"\nDistribui\u00e7\u00e3o teste:\n{y_test.value_counts()}")

In [ ]:
# Cel 8 - Treinar CatBoost

from catboost import CatBoostClassifier, Pool

# Pools do CatBoost (indicam features categoricas)
train_pool = Pool(X_train, y_train, cat_features=CAT_FEATURES)
test_pool = Pool(X_test, y_test, cat_features=CAT_FEATURES)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    eval_metric='F1',
    loss_function='Logloss',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    use_best_model=True,
)

print("Treinando CatBoost...")
start = time.time()
model.fit(train_pool, eval_set=test_pool)
train_time = time.time() - start

print(f"\nTreino conclu\u00eddo em {train_time:.1f}s")
print(f"Melhor itera\u00e7\u00e3o: {model.get_best_iteration()}")
print(f"Melhor score: {model.get_best_score()}")

In [ ]:
# Cel 9 - Avaliar modelo

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, matthews_corrcoef, log_loss, classification_report,
    confusion_matrix, average_precision_score
)
import matplotlib.pyplot as plt

# Predicoes
y_pred = model.predict(test_pool)
y_proba = model.predict_proba(test_pool)[:, 1]

# Metricas
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "auc_roc": roc_auc_score(y_test, y_proba),
    "pr_auc": average_precision_score(y_test, y_proba),
    "mcc": matthews_corrcoef(y_test, y_pred),
    "log_loss": log_loss(y_test, y_proba),
}

print("=" * 50)
print("  RESULTADOS DO CATBOOST")
print("=" * 50)
for name, value in metrics.items():
    print(f"  {name:15s}: {value:.4f}")

print(f"\n{classification_report(y_test, y_pred, target_names=['Legit', 'Phishing'])}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Legit', 'Phishing'])
ax.set_yticklabels(['Legit', 'Phishing'])
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confus\u00e3o - CatBoost Cascata')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=16)
plt.colorbar(im)
plt.tight_layout()
plt.savefig('confusion_matrix_catboost.png', dpi=150)
plt.show()

In [ ]:
# Cel 10 - Feature importance

importances = model.get_feature_importance()
feature_names = FEATURE_COLS

# Ordenar por importancia
sorted_idx = np.argsort(importances)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(sorted_idx)), importances[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Import\u00e2ncia')
ax.set_title('Feature Importance - CatBoost Cascata')
plt.tight_layout()
plt.savefig('feature_importance_catboost.png', dpi=150)
plt.show()

print("\n--- Feature Importance (Top 10) ---")
for i in sorted_idx[::-1][:10]:
    print(f"  {feature_names[i]:20s}: {importances[i]:.2f}")

In [ ]:
# Cel 11 - Simular cascata (BERT + CatBoost)
# Mostra o ganho teorico da cascata vs BERT sozinho

# Simular: para URLs onde BERT ficaria incerto (prob entre 0.15-0.85),
# o CatBoost decide. Para as demais, BERT decide sozinho.

# Como nao temos o BERT score para cada URL do test set aqui,
# mostramos apenas as metricas isoladas do CatBoost.
# A integracao real acontece na API.

print("=" * 60)
print("  RESUMO PARA INTEGRACAO NA API")
print("=" * 60)
print(f"")
print(f"  CatBoost Accuracy:  {metrics['accuracy']:.1%}")
print(f"  CatBoost F1:        {metrics['f1_score']:.1%}")
print(f"  CatBoost AUC-ROC:   {metrics['auc_roc']:.1%}")
print(f"  CatBoost MCC:       {metrics['mcc']:.4f}")
print(f"")
print(f"  Cascata esperada:")
print(f"    - BERT resolve ~80% das URLs (prob > 0.85 ou < 0.15)")
print(f"    - CatBoost resolve os ~20% incertos")
print(f"    - Latencia mediana: ~20ms (BERT) | ~2-5s (cascata)")
print(f"")
print(f"  Modelo: catboost_cascata.cbm")
print(f"  Features: {len(FEATURE_COLS)} ({len(CAT_FEATURES)} categoricas)")
print(f"  Categoricas: {CAT_FEATURES}")

In [ ]:
# Cel 12 - Salvar modelo e metadados no Google Drive

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = Path("/content/drive/MyDrive/phishing_catboost_cascata")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Salvar modelo CatBoost (formato nativo)
MODEL_FILE = SAVE_DIR / "catboost_cascata.cbm"
model.save_model(str(MODEL_FILE))
print(f"Modelo salvo: {MODEL_FILE}")

# Salvar metadados
metadata = {
    "feature_columns": FEATURE_COLS,
    "cat_features": CAT_FEATURES,
    "metrics": metrics,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "best_iteration": model.get_best_iteration(),
    "train_time_s": train_time,
    "config": {
        "iterations": 1000,
        "learning_rate": 0.05,
        "depth": 8,
        "early_stopping_rounds": 50,
    }
}

with open(SAVE_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"Metadados salvos: {SAVE_DIR / 'metadata.json'}")

# Salvar feature columns separado (para a API carregar)
with open(SAVE_DIR / "feature_columns.json", "w") as f:
    json.dump({"feature_columns": FEATURE_COLS, "cat_features": CAT_FEATURES}, f, indent=2)
print(f"Feature columns salvas: {SAVE_DIR / 'feature_columns.json'}")

# Salvar dataset enriquecido como backup
df_enriched.to_csv(SAVE_DIR / "dataset_cascata_20k.csv", index=False)
print(f"Dataset backup salvo: {SAVE_DIR / 'dataset_cascata_20k.csv'}")

# Salvar graficos
import shutil
for png in ['confusion_matrix_catboost.png', 'feature_importance_catboost.png']:
    if Path(png).exists():
        shutil.copy(png, SAVE_DIR / png)

print(f"\nTodos os arquivos salvos em: {SAVE_DIR}")
print(f"\nProximo passo:")
print(f"  1. Baixe catboost_cascata.cbm e feature_columns.json")
print(f"  2. Copie para phishing-api/model/")
print(f"  3. Integre a cascata na API")